# Day 05 Exercises — Solutions

Answer key for `day05_exercises.ipynb`. Some questions have more than one valid way to write them — these are the reference solutions, not the only correct answers.

**Before you start:** run `day05_python_sqlite.ipynb` first — this notebook connects to the `parch_and_posey.sqlite` file it generates.

In [ ]:
%load_ext sql

import sqlite3
import pandas as pd

sqlite_conn = sqlite3.connect("parch_and_posey.sqlite")
%sql sqlite:///parch_and_posey.sqlite --alias sqlite_pp

### Day 01

**1 — SELECT + LIMIT/OFFSET.** Show the 3rd-through-7th sales reps, ordered alphabetically by name.

In [ ]:
%%sql
SELECT name FROM sales_reps ORDER BY name LIMIT 5 OFFSET 2;

**2 — DISTINCT.** How many distinct regions does the company operate in?

In [ ]:
%%sql
SELECT DISTINCT name FROM region;

**3 — ORDER BY + COUNT.** How many total web events have ever been logged?

In [ ]:
%%sql
SELECT COUNT(*) AS total_events FROM web_events;

### Day 02

**4 — WHERE + comparison operators.** Which orders sold for less than $200?

In [ ]:
%%sql
SELECT id, account_id, total_amt_usd FROM orders WHERE total_amt_usd < 200;

**5 — AND / OR.** Which orders had zero posters AND more than 50 gloss units?

In [ ]:
%%sql
SELECT id, gloss_qty, poster_qty FROM orders WHERE poster_qty = 0 AND gloss_qty > 50;

**6 — BETWEEN.** Which orders were placed during 2015?

In [ ]:
%%sql
SELECT id, occurred_at FROM orders WHERE occurred_at BETWEEN '2015-01-01' AND '2015-12-31';

**7 — IN.** Which regions are Midwest or Southeast?

In [ ]:
%%sql
SELECT * FROM region WHERE name IN ('Midwest', 'Southeast');

**8 — NULL checks.** Are there any orders with a missing (NULL) poster_qty?

In [ ]:
%%sql
SELECT COUNT(*) AS null_poster_orders FROM orders WHERE poster_qty IS NULL;

**9 — LIKE.** Which accounts have a name containing 'oil' (case-insensitive)?

In [ ]:
%%sql
SELECT name FROM accounts WHERE name LIKE '%oil%';

**10 — GROUP BY + aggregates.** What's the average order size (total_amt_usd) per account, smallest first?

In [ ]:
%%sql
SELECT account_id, AVG(total_amt_usd) AS avg_order FROM orders GROUP BY account_id ORDER BY avg_order LIMIT 5;

**11 — DATE functions.** How many orders were placed each month of 2016?

In [ ]:
%%sql
SELECT STRFTIME('%m', occurred_at) AS mo, COUNT(*) AS num_orders FROM orders WHERE STRFTIME('%Y', occurred_at) = '2016' GROUP BY mo ORDER BY mo;

### Day 03

**12 — INNER JOIN.** Combine every web event with the account it belongs to.

In [ ]:
%%sql
SELECT w.id, a.name FROM web_events w JOIN accounts a ON w.account_id = a.id LIMIT 5;

**13 — Multi-table JOIN.** Trace a web event back to the region of the account it belongs to.

In [ ]:
%%sql
SELECT w.id, a.name AS account, r.name AS region FROM web_events w JOIN accounts a ON w.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id JOIN region r ON s.region_id = r.id LIMIT 5;

**14 — JOIN + GROUP BY.** Total web events per channel.

In [ ]:
%%sql
SELECT channel, COUNT(*) AS num_events FROM web_events GROUP BY channel ORDER BY num_events DESC;

**15 — JOIN + HAVING.** Which accounts have placed more than 30 orders?

In [ ]:
%%sql
SELECT a.id, a.name, COUNT(*) AS num_orders FROM accounts a JOIN orders o ON a.id = o.account_id GROUP BY a.id, a.name HAVING COUNT(*) > 30;

**16 — LEFT JOIN + anti-join.** Which accounts have NEVER logged a single web event?

In [ ]:
%%sql
SELECT a.id, a.name FROM accounts a LEFT JOIN web_events w ON a.id = w.account_id WHERE w.id IS NULL LIMIT 5;

**17 — FULL JOIN via UNION.** Every web-event/account pairing, matched or not.

In [ ]:
%%sql
SELECT * FROM web_events LEFT JOIN accounts ON web_events.account_id = accounts.id UNION SELECT * FROM web_events RIGHT JOIN accounts ON web_events.account_id = accounts.id LIMIT 5;

**18 — CASE.** Tag every order 'Bulk' (500+ total units) or 'Standard'.

In [ ]:
%%sql
SELECT id, total, CASE WHEN total >= 500 THEN 'Bulk' ELSE 'Standard' END AS order_type FROM orders LIMIT 5;

**19 — CASE + GROUP BY.** Bucket every sales rep into a 'top' (200+ orders) or 'not' performer flag.

In [ ]:
%%sql
SELECT s.name, COUNT(*) AS num_orders, CASE WHEN COUNT(*) > 200 THEN 'top' ELSE 'not' END AS flag FROM orders o JOIN accounts a ON o.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id GROUP BY s.name ORDER BY num_orders DESC LIMIT 5;

**20 — CASE + HAVING.** Which accounts fall specifically in the '$1,000-$5,000 total spend' bracket?

In [ ]:
%%sql
SELECT a.name, SUM(o.total_amt_usd) AS total_spend FROM accounts a JOIN orders o ON a.id = o.account_id GROUP BY a.name HAVING SUM(o.total_amt_usd) BETWEEN 1000 AND 5000;

**21 — String functions.** Clean a padded region name, and build a slug from a channel name.

In [ ]:
%%sql
SELECT TRIM('   Northeast   ') AS trimmed, CONCAT(LOWER(SUBSTR('facebook', 1, 4)), '-channel') AS slug;

**22 — COALESCE / IFNULL.** Fill in a friendly label wherever a computed bucket is NULL.

In [ ]:
%%sql
SELECT COALESCE(NULL, 'no bucket') AS label;

### Day 04

**23 — Scalar subquery.** Which orders sold for less than the company-wide average?

In [ ]:
%%sql
SELECT id, total_amt_usd FROM orders WHERE total_amt_usd < (SELECT AVG(total_amt_usd) FROM orders) LIMIT 5;

**24 — Row subquery.** Each account's very FIRST web event ever (earliest occurred_at).

In [ ]:
%%sql
SELECT account_id, occurred_at, channel FROM web_events WHERE (account_id, occurred_at) IN (SELECT account_id, MIN(occurred_at) FROM web_events GROUP BY account_id) ORDER BY account_id LIMIT 5;

**25 — Derived table.** Average order value, averaged across accounts (average of each account's own average).

In [ ]:
%%sql
SELECT AVG(avg_order) AS avg_of_averages FROM (SELECT account_id, AVG(total_amt_usd) AS avg_order FROM orders GROUP BY account_id) AS t;

**26 — CTE.** Same average-of-averages question, written as a CTE.

In [ ]:
%%sql
WITH t AS (SELECT account_id, AVG(total_amt_usd) AS avg_order FROM orders GROUP BY account_id) SELECT AVG(avg_order) AS avg_of_averages FROM t;

**27 — Chained CTEs.** Which sales reps sell above the average total sales per rep?

In [ ]:
%%sql
WITH rep_totals AS (SELECT s.id, s.name, SUM(o.total_amt_usd) AS total_sales FROM orders o JOIN accounts a ON o.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id GROUP BY s.id, s.name), avg_rep AS (SELECT AVG(total_sales) AS avg_sales FROM rep_totals) SELECT * FROM rep_totals WHERE total_sales > (SELECT avg_sales FROM avg_rep) LIMIT 5;

**28 — Temporary table.** Materialize the sales rep with the FEWEST total sales, then query it twice.

In [ ]:
%%sql
CREATE TEMPORARY TABLE bottom_rep_demo AS SELECT s.id, s.name, SUM(o.total_amt_usd) AS total_sales FROM orders o JOIN accounts a ON o.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id GROUP BY s.id, s.name ORDER BY total_sales ASC LIMIT 1;

**29 — View.** A 'bottom 10 accounts by spend' view, for a churn-risk report.

In [ ]:
%%sql
DROP VIEW IF EXISTS bottom10_accounts_demo;

CREATE VIEW bottom10_accounts_demo AS SELECT a.id, a.name, SUM(o.total_amt_usd) AS total_sales FROM orders o JOIN accounts a ON o.account_id = a.id GROUP BY a.id, a.name ORDER BY total_sales ASC LIMIT 10;

**30 — Window function: running total.** Running total of gloss-paper quantity, ordered chronologically.

In [ ]:
%%sql
SELECT occurred_at, gloss_qty, SUM(gloss_qty) OVER (ORDER BY occurred_at) AS running_total FROM orders ORDER BY occurred_at LIMIT 5;

**31 — PARTITION BY.** Running total of poster_qty, reset per account.

In [ ]:
%%sql
SELECT account_id, occurred_at, SUM(poster_qty) OVER (PARTITION BY account_id ORDER BY occurred_at) AS acct_running_total FROM orders ORDER BY account_id, occurred_at LIMIT 5;

**32 — RANK / DENSE_RANK / ROW_NUMBER.** Rank sales reps by total sales, comparing how ties are handled.

In [ ]:
%%sql
WITH totals AS (SELECT s.name, SUM(o.total_amt_usd) AS total_sales FROM orders o JOIN accounts a ON o.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id GROUP BY s.name) SELECT name, total_sales, RANK() OVER (ORDER BY total_sales DESC) AS rnk, DENSE_RANK() OVER (ORDER BY total_sales DESC) AS dense_rnk, ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_num FROM totals ORDER BY total_sales DESC LIMIT 5;

**33 — Ranking + filtering.** Each sales rep's single largest managed account, by that account's total spend.

In [ ]:
%%sql
WITH account_totals AS (SELECT a.id, s.name AS rep_name, a.name AS account_name, SUM(o.total_amt_usd) AS total_spend FROM orders o JOIN accounts a ON o.account_id = a.id JOIN sales_reps s ON a.sales_rep_id = s.id GROUP BY a.id, s.name, a.name), ranked AS (SELECT rep_name, account_name, total_spend, ROW_NUMBER() OVER (PARTITION BY rep_name ORDER BY total_spend DESC) AS r FROM account_totals) SELECT rep_name, account_name, total_spend FROM ranked WHERE r = 1 ORDER BY rep_name LIMIT 5;

**34 — LAG / LEAD.** Is the number of web events per month growing or shrinking?

In [ ]:
%%sql
WITH monthly AS (SELECT STRFTIME('%Y-%m', occurred_at) AS month, COUNT(*) AS num_events FROM web_events GROUP BY month) SELECT month, num_events, LAG(num_events) OVER (ORDER BY month) AS prev_month FROM monthly ORDER BY month LIMIT 5;

**35 — Stored procedures.** Call the region-report procedure Day 04 built, for region 3. (SQLite has no stored procedures — write it as a plain Python function instead, same pattern as `day05_python_sqlite.ipynb`.)

In [ ]:
def region_sales_report(region_id):
    query = (
        "SELECT r.name AS region_name, COUNT(o.id) AS num_orders, "
        "SUM(o.total_amt_usd) AS total_revenue "
        "FROM region r JOIN sales_reps s ON s.region_id = r.id "
        "JOIN accounts a ON a.sales_rep_id = s.id "
        "JOIN orders o ON o.account_id = a.id "
        "WHERE r.id = :region_id GROUP BY r.name"
    )
    return pd.read_sql(query, sqlite_conn, params={"region_id": region_id})

region_sales_report(3)

### Safe parameter binding

**36 — Bound parameters.** Write a function `accounts_in_region(region_name)` that returns every account in a given region, using a bound parameter (`:region_name`) rather than an f-string. Confirm it still returns the correct, safe result even if called with `"Midwest' OR '1'='1"`.

In [ ]:
def accounts_in_region(region_name):
    query = (
        "SELECT a.name FROM accounts a "
        "JOIN sales_reps s ON a.sales_rep_id = s.id "
        "JOIN region r ON s.region_id = r.id "
        "WHERE r.name = :region_name"
    )
    return pd.read_sql(query, sqlite_conn, params={"region_name": region_name})

accounts_in_region("Midwest").head()

In [ ]:
# a malicious-looking input — still returns the same safe result: zero matching rows,
# not every account in the table
accounts_in_region("Midwest' OR '1'='1")